# Chapter 18: SQL for Analysts

The programs from Chapter 18 of *Data Analysis with Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name.

Run the cells from top to bottom (**Run > Run All Cells**). The programs read the datasets from `../data/`, so keep this notebook in its chapter folder.

## A Database in Memory

`sql_basics.py`

In [1]:
import sqlite3

import pandas as pd

conn = sqlite3.connect(":memory:")  # a database that lives in memory
pd.read_csv("../data/orders.csv").to_sql("orders", conn, index=False)
pd.read_csv("../data/products.csv").to_sql("products", conn, index=False)


def sql(query):
    print(pd.read_sql(query, conn))
    print()


sql("""
SELECT category, COUNT(*) AS orders, ROUND(AVG(quantity), 2) AS avg_qty
FROM orders
WHERE channel = 'app'
GROUP BY category
HAVING COUNT(*) > 300
ORDER BY orders DESC
""")

sql("""
SELECT o.product,
       ROUND(SUM(o.quantity * (o.unit_price - p.unit_cost)), 0) AS profit
FROM orders AS o
JOIN products AS p ON p.product = o.product
GROUP BY o.product
ORDER BY profit DESC
LIMIT 3
""")

sql("""
SELECT * FROM (
  SELECT region, product, SUM(quantity) AS units,
         RANK() OVER (PARTITION BY region
                      ORDER BY SUM(quantity) DESC) AS rnk
  FROM orders
  GROUP BY region, product
) WHERE rnk = 1
""")

   category  orders  avg_qty
0   Bedding     438     1.56
1   Kitchen     361     1.63
2  Lighting     346     1.64
3    Garden     319     1.58
4     Decor     303     1.56

         product   profit
0     Floor Lamp  37749.0
1    Duvet Cover  24677.0
2  Cast Iron Pan  17515.0

  region        product  units  rnk
0   East     Chef Knife    191    1
1  North    Pillow Pair    232    1
2  South  Throw Blanket    170    1
3   West     Chef Knife    191    1

